Calculate a matrix-vector multiplication with OpenCL.

In [ ]:
!pip install pyopencl

In [ ]:
import numpy as np
import pyopencl as cl
import pyopencl.array as cl_array
import time

In [ ]:
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
print("Platform name:", platform.name)
print("Device name:", device.name)
print("Maximum work group size:", device.max_work_group_size)

In [ ]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)

In the following, we wish to divide the data in smaller workgroups. For this purpose, we will define the workgroup size and the number of data points. Later, the program can only be used when the workgroup size is a divisor of the number of data points. If this is not the case, one needs to pad the data array with dummy variables.

In [ ]:
workgroup_size = 2**6
n_workgroups = 8
matrix_size = workgroup_size * n_workgroups
print("Workgroup size:", workgroup_size)
print("Matrix size:", matrix_size)

There is no easy native option to use multidimensional arrays in C. One could use libraries or nested arrays. Here, let use a flattened matrix as an input vector, stored row-by-row.

In [ ]:
kernel = """
__kernel void matvec(__global const double *matrix,
                     __global const double *vector,
                     __global double *output)
{
  int dim = get_global_size(0);
  int global_id = get_global_id(0);

  double local_sum = 0.0;
  int offset = global_id * dim;
  for(int i = 0; i < dim; i++){
    local_sum += matrix[i + offset] * vector[i];
  }

  output[global_id] = local_sum;
}
"""

In [ ]:
prg = cl.Program(ctx, kernel).build()

In [ ]:
np.random.seed(0)
random_matrix = np.random.rand(matrix_size, matrix_size)
random_vector = np.random.rand(matrix_size)

In [ ]:
## Calculate the matvec with Numpy.
start = time.time()
matvec_np = random_matrix @ random_vector
time_np = time.time() - start

In [ ]:
## Calculate the matvec with OpenCL, in single precision.

start = time.time()

cl_matrix = cl_array.to_device(queue, random_matrix.ravel()) # Aplanamos la matriz para que sea un solo vector
cl_vector = cl_array.to_device(queue, random_vector)
cl_output = cl_array.empty(queue, matrix_size, dtype=np.float64)

event = prg.matvec(queue,
                   (matrix_size,), (workgroup_size,),
                   cl_matrix.data, cl_vector.data, cl_output.data)

matvec_cl = cl_output.get()

time_cl = time.time() - start
time_cl_kernel = 1e-9 * (event.profile.end - event.profile.start)

In [ ]:
print(matvec_np[:5])
print(matvec_cl[:5])

In [ ]:
print("Elapsed time Numpy :", time_np)
print("Elapsed time OpenCL:", time_cl)
print(" of which in kernel:", time_cl_kernel)